![Astrofisica Computacional](../new_logo.png)

# Computational Astrophysics – `Astroquery` II

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.



### About this notebook

In this worksheet we will present an example of the use of `Astroquery` to get the spectrum of an object from the SDSS database. 

---

---

## Obtaining an spectrum from the SDSS

First, we import the functions we need,

In [ ]:
import matplotlib 
#matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline 

# Python standard-libraries to download data from the web
from urllib.parse import urlencode
from urllib.request import urlretrieve

#Some astropy submodules that you know already
from astropy import units as u
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy.io import fits

#only here to display images
from IPython.display import Image

# These are the new modules for this notebook
from astroquery.simbad import Simbad
from astroquery.sdss import SDSS

We will work with the galaxy [NGC5406](http://simbad.u-strasbg.fr/simbad/sim-id?Ident=NGC+5406)

In [ ]:
galaxy_name = 'NGC5406'

# Information from the name of the galaxy in the SkyCoord server
galaxy = SkyCoord.from_name(galaxy_name)
galaxy

In [ ]:
# Get the coordinates of the galaxy
pos = coords.SkyCoord(galaxy.ra, galaxy.dec, frame='icrs')
pos

### Image of the galaxy

We will get the image of the galaxy from the [SDSS DR12 cutout service](http://skyservice.pha.jhu.edu/sdsscutout/)

Ypu can find information about the APIs in the skyserver at [https://skyserver.sdss.org/dr18/Support/Api](https://skyserver.sdss.org/dr18/Support/Api)

In [ ]:
import requests
import PIL 
#from PIL import Image

im_size = 3*u.arcmin # get a 3 arcmin square
im_pixels = 1024

# Define the URL to dowload the cutout
#cutoutbaseurl = 'http://skyservice.pha.jhu.edu/DR12/ImgCutout/getjpeg.aspx'
cutoutbaseurl ='http://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg'
query_string = urlencode(dict(ra=galaxy.ra.deg,
                              dec=galaxy.dec.deg,
                              width=im_pixels, height=im_pixels,
                              scale=im_size.to(u.arcsec).value/im_pixels))
url = cutoutbaseurl + '?' + query_string

# Download the image
image_name = galaxy_name+'_SDSS_cutout.jpg'
#urlretrieve(url, image_name)
img = PIL.Image.open(requests.get(url, stream = True).raw)
img.save(image_name)

In [ ]:
#load the .jpg image into the notebook
Image(image_name) 

### Data from the SDSS

Now we will get the identification numbers to grab the data from SDSS,

In [ ]:
xid = SDSS.query_region(pos, spectro=True,radius=2.5*u.arcmin)
xid

We obtain the spectra of the object from the SDSS

In [ ]:
spectra = SDSS.get_spectra(matches=xid)
# List of results
spectra

`spectra[0]` stores all the files related to the spectra for the object of interest. This is actually an array of several HDU in the FITS format

In [ ]:
spectra[0]

The spectrum is stored as a table in the component [0][1] of this object. That means that we can get the Table doing the following

In [ ]:
spectra_data = spectra[0][1].data
spectra_data

Note the structure of that table. Pay attention to the field dtype (data type). It tells you the name of the different columns available in that table.

Please also check the documentation so that you can see what are the units https://data.sdss.org/datamodel/files/BOSS_SPECTRO_REDUX/RUN2D/spectra/PLATE4/spec.html

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(10**spectra_data['loglam'], spectra_data['flux'])
plt.xlabel('wavelenght [Angstrom]')
plt.ylabel('flux [nanomaggies]')
plt.title('SDSS spectra of '+galaxy_name)
plt.show()

The component [0][3] of the object 'spectra' contains the positions of some emission lines

In [ ]:
lines = spectra[0][3].data
lines

The lines in this spectrum are

In [ ]:
lines['LINENAME']

We will plot three of these lines: '[O_II] 3727', '[O_III] 5007' and 'H_alpha'. The corresponding wavelengths are

In [ ]:
for n in ['[O_II] 3727', '[O_III] 5007', 'H_alpha']:
    print(n, " ->", lines['LINEWAVE'][lines['LINENAME']==n])

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(10**spectra_data['loglam'], spectra_data['flux'], color='black')
plt.axvline(x=lines['LINEWAVE'][lines['LINENAME']=='[O_II] 3727'], label=r'O[II]', color='cornflowerblue')
plt.axvline(x=lines['LINEWAVE'][lines['LINENAME']=='[O_III] 5007'], label=r'O[III]', color='crimson')
plt.axvline(x=lines['LINEWAVE'][lines['LINENAME']=='H_alpha'], label=r'H$\alpha$', color='green')

plt.xlabel('wavelenght [Angstrom]')
plt.ylabel('flux [nanomaggies]')
plt.title('SDSS spectra of '+galaxy_name)
plt.legend()
plt.show()

### Photometric image

Now we will obtain the photometric information of the galaxy.  We can get the images in the different SDSS bands (u,g,r,i,z)

The documentation describing the imaging data is here: https://data.sdss.org/datamodel/files/BOSS_PHOTOOBJ/frames/RERUN/RUN/CAMCOL/frame.html

In [ ]:
# Get the photometric inoformation in the g-band
images = SDSS.get_images(matches=xid, band='g')
images

The component [0][0] of the object 'images' contains the photometric information,

In [ ]:
image_data =  images[0][0].data

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image_data)
plt.colorbar()
plt.show()

The galaxy is not shown because the flux values in some of the pixels are very high compared to the typical flux.

To see the image, we will create a 'clipped image'. This will make that any pixel with a flux larger than 1 will be set 'normalized' to 1.

In [ ]:
clipped_image = image_data.copy()
clipped_image[clipped_image>1.0]=1.0

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(clipped_image)
plt.colorbar()
plt.show()

Now the galaxy is shown. Now we will crop the image to the region where the galaxy is located,

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(clipped_image[125:475,750:1100])
plt.colorbar()
plt.show()

A similar result can be obtained from the original data by taking the logarithm of the flux.

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(np.log10(image_data[125:475,750:1100]))
plt.colorbar()
plt.show()

The missing points appear due to negative values in the image data. We can modify the image by using the minimum value,

In [ ]:
image_data2 = image_data - np.min(image_data)
plt.figure(figsize=(10,10))
plt.imshow(np.log10(image_data2[125:475,750:1100]))
plt.colorbar()
plt.show()

#### Exercise 1

Identify at least three of the lines seen in the spectrum of NGC5406. Using that information, what is the redshift of this galaxy?

#### Exercise 2

Download the FITS images in the filters u,g,r,z,i for NGC5406. Plot in five different panels the logarithm of the flux in each of the bands (always cropping around the galaxy). Why do the images look different?

#### Exercise 3

Compute the radial flux profile of NGC5406. To do that take 10 different lines starting from the center of the galaxy and plot the flux as a function of radius for those ten lines. What kind of function should fit your results?

#### Exercise 4

Repeat the same producedure of this notebook (including the exercises 2 and 3) for the galaxy `SDSS J013755.71+010004.9`.

Why does this galaxy look so different from NGC5406?
